# 03. 가상 자료로 벌점(penalty) 바꿔 보기

실제 자료 없이 실행됩니다. 작은 가상 자료를 만들고, 연구에 쓴 것과 같은 풀이 코드(`r1_solvers.py`)로 벌점 두 개를 바꿔 봅니다.

- **η (soft prior):** 경로 밖 방향을 얼마나 더 세게 누를지. 0이면 보통 Ridge, 아주 크면 경로 방향만 남습니다(hard).
- **τ (약물 공유):** 약물들의 계수가 소수의 공통 방향을 쓰도록 누르는 강도. 클수록 해의 rank가 낮아집니다.

가상 세계를 두 개 만듭니다. **세계 A**는 신호가 경로 안에 있고, **세계 B**는 신호가 경로 밖에 있습니다. 실제 자료가 어느 쪽에 가까운지는 01 노트북의 결과가 말해 줍니다.

In [ ]:
import os, sys
root = os.getcwd()
while not os.path.isdir(os.path.join(root, "configs", "softprior_multitask_r1")):
    parent = os.path.dirname(root); assert parent != root; root = parent
sys.path.insert(0, os.path.join(root, "src", "softprior_multitask_r1"))
import numpy as np, matplotlib.pyplot as plt
import r1_solvers as S
rs = np.random.RandomState(0)
N, N_TEST, G, D, K = 100, 400, 200, 30, 6
sets = [np.sort(rs.choice(G, 20, replace=False)) for _ in range(K)]     # 가상 "경로" 6개, 유전자 20개씩
X, X_test = rs.normal(size=(N, G)), rs.normal(size=(N_TEST, G))
M0 = np.zeros((G, K))
for k, idx in enumerate(sets): M0[idx, k] = 1 / len(idx)
Qtrue = S.orth_basis(M0)
shared = rs.normal(size=(2, D))                                          # 약물들이 공유하는 방향 2개
W_A = Qtrue[:, :2] @ shared * 2.0                                        # 세계 A: 신호가 경로 방향에 있음
out_dir = rs.normal(size=(G, 2)); out_dir -= Qtrue @ (Qtrue.T @ out_dir); out_dir /= np.linalg.norm(out_dir, axis=0)
W_B = out_dir @ shared * 2.0                                             # 세계 B: 신호가 경로 밖에 있음
def world(W):
    return X @ W + rs.normal(size=(N, D)), X_test @ W
print("표본", N, "유전자", G, "약물", D, "경로", K)

## 풀이 함수

연구 코드와 같은 순서입니다. 학습 자료만으로 표준화하고, 경로 부분공간 Q를 만들고, 축약 공간에서 풉니다.

In [ ]:
def fit_predict(Y, sets, ratio, lam, tau_frac):
    """ratio = η/λ (np.inf면 hard), tau_frac = τ/τ_max. 시험 예측과 해의 rank를 돌려줍니다."""
    wt = np.full(N, 1 / N)
    mu, sd = S.wmean_sd(X, wt); Xs, Xt = S.standardise(X, mu, sd), S.standardise(X_test, mu, sd)
    ymu, ysd = S.wmean_sd(Y, wt); Z = (Y - ymu) / ysd
    Q = S.orth_basis(S.membership_std(sets, sd))
    if np.isinf(ratio):                       # hard: 경로 방향만 사용
        Xs, Xt, Q, ratio = Xs @ Q @ Q.T, Xt @ Q @ Q.T, Q, 0.0
    B = S.reduced_basis(Xs, Q); XB, XtB = Xs @ B, Xt @ B
    H, Rm = S.smooth_terms(XB, wt, Z, Q.shape[1], lam, ratio * lam)
    if tau_frac == 0:
        C = np.linalg.solve(H, Rm); rank = min(C.shape)
    else:
        C, info = S.admm_nuclear(H, Rm, tau_frac * S.tau_max(XB, wt, Z)); rank = info["rank"]
    return (XtB @ C) * ysd + ymu, rank

def test_error(Y, truth, sets, ratio, lam, tau_frac):
    P, rank = fit_predict(Y, sets, ratio, lam, tau_frac)
    return float(np.sqrt(((P - truth) ** 2).mean()) / truth.std()), rank     # 1.0 = 아무것도 배우지 못함

## 1. η를 바꾸기: hard, soft, prior 없음

λ는 고정하고 η/λ만 바꿉니다. 맨 오른쪽(∞)이 hard입니다.

In [ ]:
ratios = [0, 1, 10, 100, 1000, np.inf]
YA, TA = world(W_A); YB, TB = world(W_B)
errA = [test_error(YA, TA, sets, r, 0.01, 0)[0] for r in ratios]
errB = [test_error(YB, TB, sets, r, 0.01, 0)[0] for r in ratios]
print("eta/lambda:", ratios)
print("세계 A(신호가 경로 안):", np.round(errA, 3))
print("세계 B(신호가 경로 밖):", np.round(errB, 3))
fig, ax = plt.subplots(figsize=(5.5, 3.2)); xs = np.arange(len(ratios))
ax.plot(xs, errA, "o-", color="#2a78d6", label="world A: signal inside pathways")
ax.plot(xs, errB, "o-", color="#eb6834", label="world B: signal outside pathways")
ax.set_xticks(xs); ax.set_xticklabels(["0\n(Ridge)", "1", "10", "100", "1000", "inf\n(hard)"]); ax.set_xlabel("eta / lambda"); ax.set_ylabel("test error / signal SD")
ax.legend(frameon=False); plt.tight_layout(); plt.show()

오차는 신호의 표준편차로 나눈 값입니다. 1.0이면 아무것도 배우지 못한 것입니다. 세계 A에서는 η를 키울수록 좋아집니다. 세계 B에서는 hard가 신호를 통째로 잃습니다. soft prior의 장점은 **η를 자료로 고를 수 있다**는 것입니다. 세계 B에서는 η=0을 고르면 Ridge로 돌아갑니다. 다만 그때 soft prior가 Ridge보다 좋아지는 것은 아닙니다.

## 2. 경로 이름표를 섞으면

세계 A에서 유전자 이름표를 섞은 무작위 '경로'를 prior로 쓰면 이득이 사라져야 합니다. 실제 자료에서도 같은 대조를 20번 했습니다.

In [ ]:
real = test_error(YA, TA, sets, 100, 0.01, 0)[0]
perm = []
for s in range(10):
    p = np.random.RandomState(100 + s).permutation(G)
    perm.append(test_error(YA, TA, [np.sort(p[idx]) for idx in sets], 100, 0.01, 0)[0])
print("실제 경로 prior: %.3f / 섞은 경로 prior 10개: 평균 %.3f (최소 %.3f, 최대 %.3f) / prior 없음: %.3f" % (real, np.mean(perm), min(perm), max(perm), errA[0]))

## 3. τ를 바꾸기: 약물 사이 공유

두 세계 모두 약물 30개가 방향 2개를 공유하도록 만들었습니다. τ를 키우면 해의 rank가 내려갑니다. 적당한 τ에서 오차가 조금 줄고, 너무 크면 계수 전체가 0 쪽으로 눌려 다시 나빠집니다. 이득의 크기는 자료에 달려 있습니다.

In [ ]:
fracs = [0, 0.03, 0.1, 0.2, 0.3, 0.5]
res = [test_error(YB, TB, sets, 0, 0.01, f) for f in fracs]
for f, (e, r) in zip(fracs, res): print("tau/tau_max = %.2f  rank %2d  상대 오차 %.3f" % (f, r, e))
fig, ax = plt.subplots(figsize=(5.5, 3.2)); ax.plot(fracs, [e for e, _ in res], "o-", color="#1baf7a")
for f, (e, r) in zip(fracs, res): ax.annotate("rank %d" % r, (f, e), textcoords="offset points", xytext=(0, 6), ha="center", fontsize=8)
ax.set_xlabel("tau / tau_max"); ax.set_ylabel("test error / signal SD (world B)"); plt.tight_layout(); plt.show()

## 4. 공유할 것이 없는 세계

약물마다 계수가 서로 무관하면 τ는 도움이 되지 않습니다. 실제 자료에서 공유의 이득이 작았다면, 약물별로 따로 맞춘 Ridge가 이미 필요한 정보를 대부분 쓰고 있었다는 뜻일 수 있습니다.

In [ ]:
W_C = rs.normal(size=(G, D)) * 0.2                                    # 약물마다 무관한 계수
YC, TC = world(W_C)
for f in [0, 0.1, 0.3, 0.5]:
    e, r = test_error(YC, TC, sets, 0, 0.01, f); print("tau/tau_max = %.2f  rank %2d  상대 오차 %.3f" % (f, r, e))

## 정리

- hard prior는 신호가 경로 밖에 있을 때 크게 잃습니다. soft prior는 그 위험을 없애지만, 경로가 맞지 않으면 이득도 없습니다.
- 무작위 경로와의 비교가 'prior가 실제로 기여했는가'를 가릅니다.
- 약물 공유는 약물들이 정말로 방향을 공유할 때만 돕습니다.
- 이 가상 자료는 벌점의 동작을 보여 줄 뿐, 실제 간암 자료의 결론이 아닙니다.